# 17 · Final submit.zip end-to-end preflight

최종 `stage1a7_stage3_v5c.zip` 자체를 다시 풀어 실제 공개 미디어로 Stage 1/2/3 추론을 실행합니다.


In [1]:
from __future__ import annotations

from google.colab import drive
from pathlib import Path
import hashlib
import importlib.util
import json
import shutil
import time
import zipfile

import cv2
import numpy as np
import pandas as pd
import torch

# ============================================================
# 0. Paths
# ============================================================
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")

SUBMIT_ZIP = (
    DRIVE_ROOT
    / "submissions/stage1a7_stage3_v5c/stage1a7_stage3_v5c.zip"
)
A7_ZIP = DRIVE_ROOT / "submissions/stage1_a7.zip"

baseline_candidates = [
    DRIVE_ROOT / "Baseline.zip",
    Path("/content/Baseline.zip"),
]
BASELINE_ZIP = next((p for p in baseline_candidates if p.is_file()), None)

assert SUBMIT_ZIP.is_file(), SUBMIT_ZIP
assert A7_ZIP.is_file(), A7_ZIP
assert BASELINE_ZIP is not None, (
    "Baseline.zip not found. Put it at "
    "/content/drive/MyDrive/Blackbox-Detection/Baseline.zip"
)

WORK = Path("/content/stage3_v5c_e2e_preflight")
if WORK.exists():
    shutil.rmtree(WORK)

SUBMIT_DIR = WORK / "submit"
A7_DIR = WORK / "a7"
BASELINE_DIR = WORK / "baseline"
SMOKE_DIR = WORK / "smoke"

for p in (SUBMIT_DIR, A7_DIR, BASELINE_DIR, SMOKE_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("submit  :", SUBMIT_ZIP)
print("A7      :", A7_ZIP)
print("baseline:", BASELINE_ZIP)


# ============================================================
# 1. ZIP integrity / exact contract
# ============================================================
def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

with zipfile.ZipFile(SUBMIT_ZIP) as zf:
    assert zf.testzip() is None, "submit ZIP CRC failure"
    names = [n for n in zf.namelist() if n and not n.endswith("/")]

    junk = [
        n for n in names
        if "__pycache__/" in n
        or n.endswith(".pyc")
        or n.endswith(".pyo")
        or n.endswith(".DS_Store")
    ]
    assert not junk, f"junk/cache files in submit ZIP: {junk[:20]}"

    roots = {n.split("/", 1)[0] for n in names}
    assert roots <= {"inference.py", "requirements.txt", "model"}, roots

    required = {
        "inference.py",
        "requirements.txt",
        "model/stage1/model.ts",
        "model/stage2/best.pt",
        "model/stage2/resnet18-f37072fd.pth",
        "model/stage3/model.ts",
        "model/stage3/target_stats.json",
        "model/stage3/calibration.json",
        "model/stage3/v5c_selection.json",
    }
    missing = sorted(required - set(names))
    assert not missing, f"missing submit assets: {missing}"

    zf.extractall(SUBMIT_DIR)

with zipfile.ZipFile(A7_ZIP) as zf:
    assert zf.testzip() is None, "A7 ZIP CRC failure"
    zf.extractall(A7_DIR)

with zipfile.ZipFile(BASELINE_ZIP) as zf:
    assert zf.testzip() is None, "Baseline ZIP CRC failure"
    zf.extractall(BASELINE_DIR)

print("ZIP CRC/integrity: PASS")
print("ZIP top-level/junk contract: PASS")


# ============================================================
# 2. Resolve extracted roots
# ============================================================
def resolve_root(root: Path, *, need_model: bool = False) -> Path:
    candidates = [root] + [p for p in root.iterdir() if p.is_dir()]
    for p in candidates:
        if (p / "inference.py").is_file():
            if need_model and not (p / "model").is_dir():
                continue
            return p
    raise RuntimeError(f"could not resolve root under {root}")

SUBMIT_ROOT = resolve_root(SUBMIT_DIR, need_model=True)
A7_ROOT = resolve_root(A7_DIR, need_model=True)

baseline_roots = [
    p for p in [BASELINE_DIR, *[x for x in BASELINE_DIR.iterdir() if x.is_dir()]]
    if (p / "data").is_dir()
]
assert len(baseline_roots) == 1, baseline_roots
BASELINE_ROOT = baseline_roots[0]

print("submit root  :", SUBMIT_ROOT)
print("A7 root      :", A7_ROOT)
print("baseline root:", BASELINE_ROOT)


# ============================================================
# 3. Re-check Stage 1 / Stage 2 bytes against original A7
# ============================================================
for rel in (
    "model/stage1/model.ts",
    "model/stage2/best.pt",
    "model/stage2/resnet18-f37072fd.pth",
    "requirements.txt",
):
    lhs = sha256_file(SUBMIT_ROOT / rel)
    rhs = sha256_file(A7_ROOT / rel)
    assert lhs == rhs, f"A7 asset changed: {rel}"

# inference.py must be exactly identical up to the Stage 3 boundary.
marker = (
    "# ---------------------------------------------------------------------------\n"
    "# Stage 3:"
)
submit_inf = (SUBMIT_ROOT / "inference.py").read_text(encoding="utf-8")
a7_inf = (A7_ROOT / "inference.py").read_text(encoding="utf-8")
assert marker in submit_inf and marker in a7_inf
assert submit_inf.split(marker, 1)[0] == a7_inf.split(marker, 1)[0]

print("A7 Stage 1/2 assets exact match: PASS")
print("A7 Stage 1/2 inference prefix exact match: PASS")


# ============================================================
# 4. Import inference.py from the ACTUAL final ZIP
# ============================================================
def load_module(path: Path, name: str):
    spec = importlib.util.spec_from_file_location(name, path)
    assert spec is not None and spec.loader is not None
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

final_inf = load_module(
    SUBMIT_ROOT / "inference.py",
    "final_submit_inference",
)
a7_inf_module = load_module(
    A7_ROOT / "inference.py",
    "a7_reference_inference",
)

for fn in ("predict_stage1", "predict_stage2", "predict_stage3"):
    assert callable(getattr(final_inf, fn, None)), fn

print("Final inference.py import/function contract: PASS")


# ============================================================
# 5. Build small REAL-MEDIA smoke inputs from official Baseline.zip
# ============================================================
# Stage 1: one ORIGINAL + one RERECORDED.
s1_in = SMOKE_DIR / "stage1/videos"
s1_in.mkdir(parents=True, exist_ok=True)

s1_sources = [
    (
        BASELINE_ROOT / "data/stage1/original/000001.mp4",
        s1_in / "S1_O_001.mp4",
    ),
    (
        BASELINE_ROOT / "data/stage1/rerecorded/000001.mp4",
        s1_in / "S1_R_001.mp4",
    ),
]
for src, dst in s1_sources:
    assert src.is_file(), src
    shutil.copy2(src, dst)

# Stage 2: one official example video -> all original frames, keeping frame ids.
s2_video = BASELINE_ROOT / "data/stage2/videos/000001.mp4"
assert s2_video.is_file(), s2_video

s2_frame_dir = SMOKE_DIR / "stage2/images/S2_001"
s2_frame_dir.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(s2_video))
assert cap.isOpened(), s2_video
frame_index = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    ok_write = cv2.imwrite(
        str(s2_frame_dir / f"frame_{frame_index:06d}.jpg"),
        frame,
    )
    assert ok_write
    frame_index += 1
cap.release()
assert frame_index > 0
print("Stage 2 smoke frames:", frame_index)

# Stage 3: make a short 10-Hz-like clip from the public source by taking
# every second raw frame. 64 frames are enough to exercise multiple
# T=32 / stride=8 overlap windows and the final post-processing path.
s3_source = BASELINE_ROOT / "data/stage3/videos/OPEN_001.mp4"
assert s3_source.is_file(), s3_source

s3_in = SMOKE_DIR / "stage3/videos"
s3_in.mkdir(parents=True, exist_ok=True)
s3_smoke = s3_in / "OPEN_001.avi"

cap = cv2.VideoCapture(str(s3_source))
assert cap.isOpened(), s3_source
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    str(s3_smoke),
    cv2.VideoWriter_fourcc(*"MJPG"),
    10.0,
    (width, height),
)
assert writer.isOpened(), "could not open Stage 3 smoke VideoWriter"

raw_i = 0
kept = 0
while kept < 64:
    ok, frame = cap.read()
    if not ok:
        break
    if raw_i % 2 == 0:
        writer.write(frame)
        kept += 1
    raw_i += 1

cap.release()
writer.release()

assert kept == 64, f"expected 64 Stage3 frames, got {kept}"
print("Stage 3 smoke frames:", kept)
print("Real-media smoke inputs: PASS")


# ============================================================
# 6. Run FINAL ZIP inference end-to-end
# ============================================================
def timed(label, fn):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    out = fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    peak_gib = (
        torch.cuda.max_memory_allocated() / 2**30
        if torch.cuda.is_available()
        else 0.0
    )
    print(
        f"{label}: {elapsed:.2f}s | "
        f"peak allocated {peak_gib:.2f} GiB"
    )
    return out, elapsed

final_s1, t_s1 = timed(
    "predict_stage1(final)",
    lambda: final_inf.predict_stage1(
        SMOKE_DIR / "stage1",
        SUBMIT_ROOT / "model/stage1",
    ),
)

final_s2, t_s2 = timed(
    "predict_stage2(final)",
    lambda: final_inf.predict_stage2(
        SMOKE_DIR / "stage2",
        SUBMIT_ROOT / "model/stage2",
    ),
)

final_s3, t_s3 = timed(
    "predict_stage3(final)",
    lambda: final_inf.predict_stage3(
        SMOKE_DIR / "stage3",
        SUBMIT_ROOT / "model/stage3",
    ),
)


# ============================================================
# 7. Runtime equivalence: Stage 1 / Stage 2 must match original A7
# ============================================================
a7_s1, _ = timed(
    "predict_stage1(A7 ref)",
    lambda: a7_inf_module.predict_stage1(
        SMOKE_DIR / "stage1",
        A7_ROOT / "model/stage1",
    ),
)

a7_s2, _ = timed(
    "predict_stage2(A7 ref)",
    lambda: a7_inf_module.predict_stage2(
        SMOKE_DIR / "stage2",
        A7_ROOT / "model/stage2",
    ),
)

pd.testing.assert_frame_equal(
    final_s1.sort_values("ID").reset_index(drop=True),
    a7_s1.sort_values("ID").reset_index(drop=True),
    check_dtype=True,
)
pd.testing.assert_frame_equal(
    final_s2.sort_values("ID").reset_index(drop=True),
    a7_s2.sort_values("ID").reset_index(drop=True),
    check_dtype=True,
)

print("Stage 1 runtime equivalence vs A7: PASS")
print("Stage 2 runtime equivalence vs A7: PASS")


# ============================================================
# 8. Output schema / value / coverage checks
# ============================================================
assert list(final_s1.columns) == ["ID", "answer"]
assert len(final_s1) == 2
assert set(final_s1["ID"]) == {"S1_O_001", "S1_R_001"}
assert set(final_s1["answer"]) <= {"ORIGINAL", "RERECORDED"}
assert not final_s1.isna().any().any()

assert list(final_s2.columns) == [
    "ID",
    "collision_frame",
    "entry_frame",
    "evasion_space",
    "entry_side",
]
assert len(final_s2) == 1
assert final_s2.iloc[0]["ID"] == "S2_001"
assert final_s2.iloc[0]["entry_side"] in {"LEFT", "RIGHT"}
assert not final_s2.isna().any().any()

assert list(final_s3.columns) == [
    "ID",
    "sample_index",
    "accel_label",
    "steer_label",
]
assert len(final_s3) == 64, len(final_s3)
assert final_s3["ID"].nunique() == 1
assert final_s3["ID"].iloc[0] == "OPEN_001"
np.testing.assert_array_equal(
    final_s3["sample_index"].to_numpy(),
    np.arange(64, dtype=np.int64),
)
assert set(final_s3["accel_label"]) <= {
    "ACCELERATING",
    "DECELERATING",
    "CONSTANT",
    "STOPPED",
}
assert set(final_s3["steer_label"]) <= {
    "LEFT",
    "STRAIGHT",
    "RIGHT",
}
assert not final_s3.isna().any().any()

print("Stage 1 output schema/value check: PASS")
print("Stage 2 output schema/value check: PASS")
print("Stage 3 output schema/value/coverage check: PASS")


# ============================================================
# 9. Report
# ============================================================
report = {
    "submit_zip": str(SUBMIT_ZIP),
    "submit_sha256": sha256_file(SUBMIT_ZIP),
    "stage1_rows": int(len(final_s1)),
    "stage2_rows": int(len(final_s2)),
    "stage3_rows": int(len(final_s3)),
    "stage1_seconds": float(t_s1),
    "stage2_seconds": float(t_s2),
    "stage3_seconds_64_frames": float(t_s3),
    "stage3_accel_counts": {
        str(k): int(v)
        for k, v in final_s3["accel_label"].value_counts().items()
    },
    "stage3_steer_counts": {
        str(k): int(v)
        for k, v in final_s3["steer_label"].value_counts().items()
    },
}

REPORT_PATH = (
    DRIVE_ROOT
    / "submissions/stage1a7_stage3_v5c/e2e_preflight_report.json"
)
REPORT_PATH.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("\n--- FINAL OUTPUT PREVIEW ---")
print("\nStage 1")
display(final_s1)
print("\nStage 2")
display(final_s2)
print("\nStage 3")
display(final_s3.head(10))
print("...")
display(final_s3.tail(5))

print("\nFINAL E2E PREFLIGHT: PASS")
print("Report:", REPORT_PATH)
print("Submit SHA256:", report["submit_sha256"])


Mounted at /content/drive
submit  : /content/drive/MyDrive/Blackbox-Detection/submissions/stage1a7_stage3_v5c/stage1a7_stage3_v5c.zip
A7      : /content/drive/MyDrive/Blackbox-Detection/submissions/stage1_a7.zip
baseline: /content/drive/MyDrive/Blackbox-Detection/Baseline.zip
ZIP CRC/integrity: PASS
ZIP top-level/junk contract: PASS
submit root  : /content/stage3_v5c_e2e_preflight/submit
A7 root      : /content/stage3_v5c_e2e_preflight/a7/stage1_a7
baseline root: /content/stage3_v5c_e2e_preflight/baseline
A7 Stage 1/2 assets exact match: PASS
A7 Stage 1/2 inference prefix exact match: PASS
Final inference.py import/function contract: PASS
Stage 2 smoke frames: 50
Stage 3 smoke frames: 64
Real-media smoke inputs: PASS
predict_stage1(final): 8.99s | peak allocated 0.49 GiB
predict_stage2(final): 1.99s | peak allocated 0.27 GiB
predict_stage3(final): 11.14s | peak allocated 0.90 GiB
predict_stage1(A7 ref): 6.27s | peak allocated 0.49 GiB
predict_stage2(A7 ref): 0.80s | peak allocated 0.27

,ID,answer
0,S1_O_001,ORIGINAL
1,S1_R_001,ORIGINAL



Stage 2


,ID,collision_frame,entry_frame,evasion_space,entry_side
0,S2_001,32,46,1,LEFT



Stage 3


,ID,sample_index,accel_label,steer_label
0,OPEN_001,0,CONSTANT,STRAIGHT
1,OPEN_001,1,CONSTANT,STRAIGHT
2,OPEN_001,2,CONSTANT,STRAIGHT
3,OPEN_001,3,CONSTANT,STRAIGHT
4,OPEN_001,4,CONSTANT,STRAIGHT
5,OPEN_001,5,CONSTANT,STRAIGHT
6,OPEN_001,6,CONSTANT,STRAIGHT
7,OPEN_001,7,CONSTANT,STRAIGHT
8,OPEN_001,8,CONSTANT,STRAIGHT
9,OPEN_001,9,CONSTANT,STRAIGHT


...


,ID,sample_index,accel_label,steer_label
59,OPEN_001,59,CONSTANT,RIGHT
60,OPEN_001,60,CONSTANT,RIGHT
61,OPEN_001,61,CONSTANT,RIGHT
62,OPEN_001,62,CONSTANT,RIGHT
63,OPEN_001,63,CONSTANT,RIGHT



FINAL E2E PREFLIGHT: PASS
Report: /content/drive/MyDrive/Blackbox-Detection/submissions/stage1a7_stage3_v5c/e2e_preflight_report.json
Submit SHA256: 95b9a2450d062c2e9f1585d46d97c68d5c32a2693cd85eb77d90656aa43cb925
